# 모기 비행 궤적 예측 AI 경진대회 베이스라인
- 4개 모델(1D-CNN, TCN, Transformer, LightGBM)과 5-Fold Stacking Ensemble 적용
- 물리 역학 기반 피처링, 3D Rotation 증강 기법 포함


In [1]:
# 1. 패키지 로드 및 설정 (Imports and Configuration)
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import Ridge
import math
import datetime
import warnings
warnings.filterwarnings('ignore')

CFG = {
    'DATA_DIR': './data',
    'SUBMIT_DIR': './submit',
    'EPOCHS': 20, 
    'BATCH_SIZE': 64,
    'LR': 1e-3,
    'SEED': 42,
    'N_FOLDS': 5,
}

os.makedirs(CFG['SUBMIT_DIR'], exist_ok=True)

def seed_everything(seed):
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(CFG['SEED'])

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


Using device: mps


In [2]:
# 2. 데이터 로드 (Data Load)
train_labels = pd.read_csv(f"{CFG['DATA_DIR']}/train_labels.csv")
train_ids = train_labels['id'].values
y_train = train_labels[['x', 'y', 'z']].values

def load_seqs(ids, folder):
    seqs = []
    for uid in ids:
        df = pd.read_csv(f"{CFG['DATA_DIR']}/{folder}/{uid}.csv")
        seqs.append(df[['x', 'y', 'z']].values)
    return np.array(seqs)

print("Loading train sequences...")
X_train = load_seqs(train_ids, 'train')

test_files = sorted(glob.glob(f"{CFG['DATA_DIR']}/test/*.csv"))
test_ids = [os.path.basename(f).split('.')[0] for f in test_files]

print("Loading test sequences...")
X_test = load_seqs(test_ids, 'test')
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Loading train sequences...
Loading test sequences...
Train shape: (10000, 11, 3)
Test shape: (10000, 11, 3)


In [3]:
# 3. 물리 역학 기반 피처 엔지니어링 (Physics-based Feature Engineering)
# 속도, 가속도, 저크 및 3차원 구면 좌표계 변환을 통한 윈도우 통계량 추출
def compute_features(seq):
    # seq: (N, 11, 3)
    
    # 1차, 2차, 3차 차분 (Velocity, Acceleration, Jerk)
    v = np.zeros_like(seq)
    v[:, 1:, :] = seq[:, 1:, :] - seq[:, :-1, :]
    
    a = np.zeros_like(seq)
    a[:, 2:, :] = v[:, 2:, :] - v[:, 1:-1, :]
    
    j = np.zeros_like(seq)
    j[:, 3:, :] = a[:, 3:, :] - a[:, 2:-1, :]
    
    # 3차원 구면 좌표계 변환
    r = np.sqrt(np.sum(seq**2, axis=-1)) # 반지름
    yaw = np.arctan2(seq[:, :, 1], seq[:, :, 0]) # Yaw
    pitch = np.arctan2(seq[:, :, 2], np.sqrt(seq[:, :, 0]**2 + seq[:, :, 1]**2)) # Pitch
    
    # 각속도
    yaw_v = np.zeros_like(yaw)
    yaw_v[:, 1:] = yaw[:, 1:] - yaw[:, :-1]
    
    pitch_v = np.zeros_like(pitch)
    pitch_v[:, 1:] = pitch[:, 1:] - pitch[:, :-1]
    
    # 윈도우 통계량 (Mean, Std, Max, Min) - x, y, z
    seq_mean = np.mean(seq, axis=1)
    seq_std = np.std(seq, axis=1)
    seq_max = np.max(seq, axis=1)
    seq_min = np.min(seq, axis=1)
    
    # 피처 병합
    seq_features = np.concatenate([
        seq, v, a, j, 
        r[..., np.newaxis], 
        yaw[..., np.newaxis], 
        pitch[..., np.newaxis],
        yaw_v[..., np.newaxis],
        pitch_v[..., np.newaxis]
    ], axis=2) # 총 3+3+3+3+1+1+1+1+1 = 17 features
    
    global_features = np.concatenate([
        seq_mean, seq_std, seq_max, seq_min
    ], axis=1) # 3*4 = 12 features
    
    return seq_features, global_features

print("Computing features...")
X_train_seq_f, X_train_global_f = compute_features(X_train)
X_test_seq_f, X_test_global_f = compute_features(X_test)

# GBDT용 피처 구성 (평탄화)
X_train_gbdt = np.concatenate([X_train_seq_f.reshape(X_train.shape[0], -1), X_train_global_f], axis=1)
X_test_gbdt = np.concatenate([X_test_seq_f.reshape(X_test.shape[0], -1), X_test_global_f], axis=1)


Computing features...


In [4]:
# 4. 파이토치 Dataset 및 데이터 증강 (Dataset & Data Augmentation)
# Z축 기준 회전(3D Rotation) 및 가우시안 노이즈 주입
class MosquitoDataset(Dataset):
    def __init__(self, raw_seq, targets=None, is_train=False):
        self.raw_seq = torch.tensor(raw_seq, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32) if targets is not None else None
        self.is_train = is_train
        
    def __len__(self):
        return len(self.raw_seq)
        
    def __getitem__(self, idx):
        seq = self.raw_seq[idx].clone()
        y = self.targets[idx].clone() if self.targets is not None else None
        
        # 증강 기법 적용
        if self.is_train:
            # Random Z-axis Rotation
            angle = torch.rand(1).item() * 2 * np.pi
            cos_a = np.cos(angle)
            sin_a = np.sin(angle)
            rot_matrix = torch.tensor([
                [cos_a, -sin_a, 0],
                [sin_a,  cos_a, 0],
                [0,      0,     1]
            ], dtype=torch.float32)
            seq = torch.matmul(seq, rot_matrix.T)
            if y is not None:
                y = torch.matmul(y, rot_matrix.T)
            
            # Gaussian Noise 주입
            noise = torch.randn_like(seq) * 0.005 
            seq += noise
            
        # 개별 샘플에 대한 피처 계산
        seq_np = seq.unsqueeze(0).numpy()
        seq_f, global_f = compute_features(seq_np)
        
        seq_f = torch.tensor(seq_f.squeeze(0), dtype=torch.float32)
        global_f = torch.tensor(global_f.squeeze(0), dtype=torch.float32)
        
        if y is not None:
            return seq_f, global_f, y
        return seq_f, global_f


In [5]:
# 5. 모델 아키텍처 정의 (Model Architectures)
# 1D-CNN, TCN, Transformer Encoder

# 5.1. 1D-CNN Model
class CNN1D(nn.Module):
    def __init__(self, num_seq_features=17, num_global_features=12):
        super().__init__()
        self.conv1 = nn.Conv1d(num_seq_features, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(64 + num_global_features, 64),
            nn.ReLU(),
            nn.Linear(64, 3)
        )
    def forward(self, seq_x, global_x):
        x = seq_x.transpose(1, 2) # (B, Channels, SeqLen)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1)
        out = self.fc(torch.cat([x, global_x], dim=1))
        return out

# 5.2. TCN Model
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super(Chomp1d, self).__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super(TemporalBlock, self).__init__()
        self.conv1 = nn.Conv1d(n_inputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(n_outputs, n_outputs, kernel_size, stride=stride, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)
        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1,
                                 self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TCNModel(nn.Module):
    def __init__(self, num_seq_features=17, num_global_features=12):
        super().__init__()
        num_channels = [32, 64]
        layers = []
        for i in range(len(num_channels)):
            dilation_size = 2 ** i
            in_channels = num_seq_features if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size=3, stride=1, dilation=dilation_size, padding=(3-1) * dilation_size, dropout=0.1)]
        self.network = nn.Sequential(*layers)
        self.fc = nn.Sequential(
            nn.Linear(num_channels[-1] + num_global_features, 64),
            nn.ReLU(),
            nn.Linear(64, 3)
        )
        
    def forward(self, seq_x, global_x):
        x = seq_x.transpose(1, 2)
        x = self.network(x)
        x = x[:, :, -1]
        out = self.fc(torch.cat([x, global_x], dim=1))
        return out

# 5.3. Transformer Encoder Model
class TransformerModel(nn.Module):
    def __init__(self, num_seq_features=17, num_global_features=12):
        super().__init__()
        self.d_model = 32
        self.embedding = nn.Linear(num_seq_features, self.d_model)
        encoder_layers = nn.TransformerEncoderLayer(d_model=self.d_model, nhead=4, dim_feedforward=64, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=2)
        self.fc = nn.Sequential(
            nn.Linear(self.d_model + num_global_features, 64),
            nn.ReLU(),
            nn.Linear(64, 3)
        )
        
    def forward(self, seq_x, global_x):
        x = self.embedding(seq_x)
        
        # Positional Encoding (device-safe)
        pe = torch.zeros(1, x.size(1), self.d_model, device=x.device)
        position = torch.arange(x.size(1), device=x.device).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, self.d_model, 2, device=x.device).float() * (-math.log(10000.0) / self.d_model))
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        x = x + pe
        
        out = self.transformer_encoder(x)
        out = out.mean(dim=1) # Global Average Pooling
        out = self.fc(torch.cat([out, global_x], dim=1))
        return out


In [6]:
# 6. 학습 및 추론 함수 (Training and Inference Helpers)
def train_model(model_class, epochs, train_loader, val_loader, device):
    model = model_class().to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=CFG['LR'])
    
    best_loss = float('inf')
    best_model_wts = None
    
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            seq_x, glob_x, y = batch[0].to(device), batch[1].to(device), batch[2].to(device)
            optimizer.zero_grad()
            out = model(seq_x, glob_x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                seq_x, glob_x, y = batch[0].to(device), batch[1].to(device), batch[2].to(device)
                out = model(seq_x, glob_x)
                val_loss += criterion(out, y).item() * seq_x.size(0)
        val_loss /= len(val_loader.dataset)
        
        if val_loss < best_loss:
            best_loss = val_loss
            best_model_wts = model.state_dict()
            
    model.load_state_dict(best_model_wts)
    return model

def predict(model, loader, device):
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            seq_x, glob_x = batch[0].to(device), batch[1].to(device)
            out = model(seq_x, glob_x)
            preds.append(out.cpu().numpy())
    return np.vstack(preds)


In [ ]:
# 안전모드: 강제 CPU 사용 및 학습 부담 축소
print('[ACTION] Applying safe-mode: force CPU and reduce training load')
import torch
device = torch.device('cpu')
print(f'Switched device -> {device}')
CFG['EPOCHS'] = 1
CFG['BATCH_SIZE'] = 8
print(f"CFG updated: EPOCHS={CFG['EPOCHS']}, BATCH_SIZE={CFG['BATCH_SIZE']}")

# CUDA 캐시 비우기 (있다면)
if torch.cuda.is_available():
    try:
        torch.cuda.empty_cache()
        print('[INFO] cuda.empty_cache() called')
    except Exception as e:
        print('[WARN] cuda.empty_cache() failed:', e)


[ACTION] Applying safe-mode: force CPU and reduce training load
Switched device -> cpu
CFG updated: EPOCHS=1, BATCH_SIZE=8


: 

In [ ]:
# 7. 5-Fold Cross Validation 및 앙상블을 위한 OOF 추출
# Quick liveness ping to check kernel responsiveness
import sys, os, time
print('[PING] kernel check start')
print('pid:', os.getpid())
print('python:', sys.version.split()[0])
print('torch version:', getattr(torch, '__version__', 'N/A'))
print('device variable:', device)
print('sleeping 0.1s to ensure prints flush...')
time.sleep(0.1)
print('[PING] kernel check end')

# 디버그: 주요 변수 상태 출력
print("[DEBUG] Checking data shapes and samples before CV")
print("CFG['N_FOLDS']:", CFG.get('N_FOLDS'))
print("X_train type/shape:", type(X_train), getattr(X_train, 'shape', 'N/A'))
print("y_train type/shape:", type(y_train), getattr(y_train, 'shape', 'N/A'))
print("X_train_gbdt type/shape:", type(X_train_gbdt), getattr(X_train_gbdt, 'shape', 'N/A'))
print("test_ids len:", len(test_ids) if 'test_ids' in globals() else 'N/A')
print("First train ids:", train_ids[:5] if 'train_ids' in globals() else 'N/A')

# 안전한 n_splits 설정
n_splits = CFG['N_FOLDS']
if isinstance(X_train, (list, tuple)):
    n_samples = len(X_train)
elif hasattr(X_train, 'shape'):
    n_samples = X_train.shape[0]
else:
    n_samples = 0

if n_samples == 0:
    print('[ERROR] No training samples found (n_samples=0). 확인: 데이터 로드 셀을 다시 실행하세요.')

if n_samples < n_splits:
    print(f"[WARN] n_samples ({n_samples}) < n_splits ({n_splits}). Adjusting n_splits to {max(1, n_samples)}")
    n_splits = max(1, n_samples)

kf = KFold(n_splits=n_splits, shuffle=True, random_state=CFG['SEED'])

oof_cnn = np.zeros_like(y_train)
oof_tcn = np.zeros_like(y_train)
oof_tf = np.zeros_like(y_train)
oof_gbdt = np.zeros_like(y_train)

test_cnn = np.zeros((len(test_ids), 3))
test_tcn = np.zeros((len(test_ids), 3))
test_tf = np.zeros((len(test_ids), 3))
test_gbdt = np.zeros((len(test_ids), 3))

test_dataset = MosquitoDataset(X_test, is_train=False)
test_loader = DataLoader(test_dataset, batch_size=CFG['BATCH_SIZE'], shuffle=False)

for fold, (trn_idx, val_idx) in enumerate(kf.split(X_train)):
    print(f"========== Fold {fold+1} ==========")
    
    # 7.1. GBDT 학습 및 추론
    print("Training LightGBM...")
    X_tr_g, y_tr_g = X_train_gbdt[trn_idx], y_train[trn_idx]
    X_va_g, y_va_g = X_train_gbdt[val_idx], y_train[val_idx]
    
    try:
        # 경량화: 트리 수 감소 및 병렬화 적용
        gbdt_model = MultiOutputRegressor(
            lgb.LGBMRegressor(n_estimators=20, n_jobs=-1, random_state=CFG['SEED'], verbose=-1),
            n_jobs=-1
        )
        gbdt_model.fit(X_tr_g, y_tr_g)
        oof_gbdt[val_idx] = gbdt_model.predict(X_va_g)
        test_gbdt += gbdt_model.predict(X_test_gbdt) / n_splits
    except Exception as e:
        print('[ERROR] LightGBM training/predict failed:', e)
        raise
    
    # 7.2. 딥러닝 모델용 데이터 로더 준비
    train_ds = MosquitoDataset(X_train[trn_idx], y_train[trn_idx], is_train=True)
    val_ds = MosquitoDataset(X_train[val_idx], y_train[val_idx], is_train=False)
    
    train_dl = DataLoader(train_ds, batch_size=CFG['BATCH_SIZE'], shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=CFG['BATCH_SIZE'], shuffle=False)
    
    # 7.3. 1D-CNN 학습
    print("Training 1D-CNN...")
    cnn = train_model(CNN1D, CFG['EPOCHS'], train_dl, val_dl, device)
    oof_cnn[val_idx] = predict(cnn, val_dl, device)
    test_cnn += predict(cnn, test_loader, device) / n_splits
    
    # 7.4. TCN 학습
    print("Training TCN...")
    tcn = train_model(TCNModel, CFG['EPOCHS'], train_dl, val_dl, device)
    oof_tcn[val_idx] = predict(tcn, val_dl, device)
    test_tcn += predict(tcn, test_loader, device) / n_splits
    
    # 7.5. Transformer 학습
    print("Training Transformer Encoder...")
    tf_m = train_model(TransformerModel, CFG['EPOCHS'], train_dl, val_dl, device)
    oof_tf[val_idx] = predict(tf_m, val_dl, device)
    test_tf += predict(tf_m, test_loader, device) / n_splits

[PING] kernel check start
pid: 9647
python: 3.13.5
torch version: 2.9.1
device variable: cpu
sleeping 0.1s to ensure prints flush...
[PING] kernel check end
[DEBUG] Checking data shapes and samples before CV
CFG['N_FOLDS']: 5
X_train type/shape: <class 'numpy.ndarray'> (10000, 11, 3)
y_train type/shape: <class 'numpy.ndarray'> (10000, 3)
X_train_gbdt type/shape: <class 'numpy.ndarray'> (10000, 199)
test_ids len: 10000
First train ids: ['TRAIN_00001' 'TRAIN_00002' 'TRAIN_00003' 'TRAIN_00004' 'TRAIN_00005']
========== Fold 1 ==========
Training LightGBM...


In [ ]:
# 8. Stacking 앙상블 (Meta-Learner 적용)
print("========== Stacking ==========")
# OOF 예측값 병합 (N, 12)
oof_meta = np.concatenate([oof_cnn, oof_tcn, oof_tf, oof_gbdt], axis=1)
test_meta = np.concatenate([test_cnn, test_tcn, test_tf, test_gbdt], axis=1)

# 메타 러너로 Ridge 회귀 사용
meta_learner = MultiOutputRegressor(Ridge(alpha=1.0))
meta_learner.fit(oof_meta, y_train)

# 메타 러너 추론
final_preds = meta_learner.predict(test_meta)

# 평가 (OOF RMSE)
stacking_rmse = np.sqrt(mean_squared_error(y_train, meta_learner.predict(oof_meta)))
print(f"Stacking Ensemble OOF RMSE: {stacking_rmse:.4f}")


In [ ]:
# 9. 최종 제출 파일 생성 (Submission)
submit = pd.DataFrame({
    'id': test_ids,
    'x': final_preds[:, 0],
    'y': final_preds[:, 1],
    'z': final_preds[:, 2]
})

now_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
submit_path = f"{CFG['SUBMIT_DIR']}/{now_str}.csv"
submit.to_csv(submit_path, index=False)
print(f"Submission saved successfully to: {submit_path}")
